In [22]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_dir = "../../data/"

target = pd.read_csv(data_dir + "train.csv")

book = pd.read_parquet(data_dir + "book_train.parquet")
book["stock_id"] = book["stock_id"].astype("int16")

trade = pd.read_parquet(data_dir + "trade_train.parquet")
trade["stock_id"] = trade["stock_id"].astype("int16")

In [23]:
# helpers
def realized_vol(series):
    return np.sqrt(np.sum(np.square(series)))

def rmspe(y_true, y_pred):
    return np.sqrt(np.mean(((y_true - y_pred) / y_true) ** 2))

def window_stats(df, col, keys=["stock_id", "time_id"], prefix = ""):
    g = df.groupby(keys, observed=True)[col]
    out = pd.DataFrame({
        "mean": g.mean(),
        "std":  g.std(ddof=0),
        # "iqr":  g.quantile(0.75) - g.quantile(0.25),
        "last": g.last(),
        "max": g.max(),
    })
    out["zscore_last"] = ((out["last"] - out["mean"]) / out["std"]).where(out["std"] > 0, 0.0)
    # out["std_mean"] = out["std"] / out["mean"]
    return out.drop(columns="last").add_prefix(f"{prefix}{col}_")

def add_market_features(df, transform, stats=("mean",), rel=True):
    fn = {"log": np.log, "log1p": np.log1p}
    out = {}
    for c, how in transform.items():
        x = fn[how](df[c]) if how else df[c]
        name = f"{how}_{c}" if how else c
        g = x.groupby(df["time_id"])
        for s in stats:
            out[f"mkt_{s}_{name}"] = g.transform(s)
        if rel:
            out[f"rel_{name}"] = x - g.transform("mean")
    return pd.concat([df, pd.DataFrame(out, index=df.index)], axis=1)


In [24]:
# pre-group
grouped = book.groupby(["stock_id", "time_id"], observed=True)
trade_grouped = trade.groupby(["stock_id", "time_id"], observed=True)

# total book size
book["tot_size"] = book["bid_size1"] + book["bid_size2"] + book["ask_size1"] + book["ask_size2"]

# WAP (3 versions)
book["wap1"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"])
             / (book["bid_size1"] + book["ask_size1"])).astype("float32")
book["wap2"] = ((book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / (book["bid_size2"] + book["ask_size2"])).astype("float32")
book["wap12"] = ((book["bid_price1"] * book["ask_size1"] + book["ask_price1"] * book["bid_size1"] + 
            book["bid_price2"] * book["ask_size2"] + book["ask_price2"] * book["bid_size2"])
             / book["tot_size"]).astype("float32")

for k in ["1", "2", "12"]:
    book[f"log_wap{k}"] = np.log(book[f"wap{k}"])

log_cols = ["log_wap1", "log_wap2", "log_wap12"]
ret_cols = ["log_ret1", "log_ret2", "log_ret12"]

# log return on WAP1
book[ret_cols] = grouped[log_cols].diff().to_numpy()

# spread
book["spread"] = book["ask_price1"] - book["bid_price1"]

# imbalance features
book["imb1"] = ((book.bid_size1 - book.ask_size1) / (book.bid_size1 + book.ask_size1)).astype("float32")
book["imb12"] = ((book.bid_size1 + book.bid_size2 - book.ask_size1 - book.ask_size2)
                 / (book.bid_size1 + book.bid_size2 + book.ask_size1 + book.ask_size2)).astype("float32")
book["imb1_abs"] = book["imb1"].abs()
book["imb12_abs"] = book["imb12"].abs()

imb_features = grouped.agg(
    imb1_mean     =("imb1", "mean"),
    imb1_std      =("imb1", "std"),
    imb1_abs_mean =("imb1_abs", "mean"),
    imb12_mean    =("imb12", "mean"),
    imb12_std     =("imb12", "std"),
    imb12_abs_mean=("imb12_abs", "mean"),
)

# first and last wap distance to 1
fwap1_to1 = (grouped["wap1"].first() - 1.).abs().rename("fwap1_to1")
lwap1_to1 = (grouped["wap1"].last() - 1.).abs().rename("lwap1_to1")

# book realized vol (3 versions)
book_rv = grouped[["log_ret1", "log_ret2", "log_ret12"]].agg(realized_vol)
book_rv.rename(columns={"log_ret1":"rv1", "log_ret2":"rv2", "log_ret12":"rv12"}, inplace=True)

# trade realized vol (1 version)
trade["log_price"] = np.log(trade["price"])
trade["log_ret"] = trade.groupby(["stock_id", "time_id"])["log_price"].diff()
trade_rv = trade.groupby(["stock_id", "time_id"], observed=True)[["log_ret"]].agg(realized_vol)
trade_rv.rename(columns={"log_ret":"trade_rv"}, inplace=True)

# trade ratio
n_quotes = grouped.size().rename("n_quotes")
n_trades = trade_grouped.size().rename("n_trades")
trade_ratio = pd.merge(n_quotes, n_trades, on = ["stock_id", "time_id"], how = "left")
trade_ratio["trade_ratio"] = trade_ratio["n_trades"] / trade_ratio["n_quotes"]
trade_ratio.fillna(0, inplace = True)

# trade total volume
trade_tot_volume = trade_grouped["size"].sum().rename("trade_tot_volume")

book.drop(columns=["log_wap1", "log_wap2", "log_wap12", "imb1_abs", "imb12_abs", "imb1", "imb12"], inplace=True)

del grouped, trade_grouped

In [5]:
# buy/sell features from trade, inferred in relevance to mid price

# 1. mid、半价差，以及上一个时刻的值（成交前的报价）
bid = book["bid_price1"].to_numpy()
ask = book["ask_price1"].to_numpy()
mid = (bid + ask) / 2
hs = (ask - bid) / 2

sid = book["stock_id"].to_numpy()
tid = book["time_id"].to_numpy()
secs = book["seconds_in_bucket"].to_numpy()
same_window = np.r_[False, (sid[1:] == sid[:-1]) & (tid[1:] == tid[:-1])]

# 前提检查：每个窗口的行连续出现（窗口开头的个数 = 窗口总数），且窗口内秒数递增
n_windows = len(pd.MultiIndex.from_arrays([sid, tid]).unique())
assert (~same_window).sum() == n_windows, "有窗口的行不连续"
assert (np.diff(secs)[same_window[1:]] > 0).all(), "窗口内秒数不是递增的"

mid_prev = np.where(same_window, np.roll(mid, 1), mid)   # 窗口第一行没有上一个时刻，用同一秒的
hs_prev = np.where(same_window, np.roll(hs, 1), hs)

# 2. 把 (stock_id, time_id, seconds) 编码成一个整数，找到每笔成交对应 book 的哪一行
def row_key(df):
    return (df["stock_id"].to_numpy().astype(np.int64) * 100_000_000
            + df["time_id"].to_numpy().astype(np.int64) * 1_000
            + df["seconds_in_bucket"].to_numpy())

pos = pd.Index(row_key(book)).get_indexer(row_key(trade))
print("找不到对应 book 行的成交数:", (pos < 0).sum())

# 3. 成交价的相对位置：-1 = bid，0 = mid，+1 = ask
tq = trade[["stock_id", "time_id", "price", "size"]].copy()
x = (tq["price"].to_numpy() - mid_prev[pos]) / hs_prev[pos]
tq["x"] = np.where(pos >= 0, x, np.nan)                  # 找不到的成交设为 NaN

THRESH = 0.5
tq["is_buy"] = tq["x"] >= THRESH
tq["is_sell"] = tq["x"] <= -THRESH
tq["buy_vol"] = tq["size"] * tq["is_buy"]
tq["sell_vol"] = tq["size"] * tq["is_sell"]
tq["abs_x"] = tq["x"].abs()
tq["sweep"] = tq["abs_x"] > 1.5          # 成交价远超出一档，说明吃穿了好几档

# 4. 按窗口聚合
side_feats = tq.groupby(["stock_id", "time_id"], observed=True).agg(
    n_buy=("is_buy", "sum"),
    n_sell=("is_sell", "sum"),
    buy_vol=("buy_vol", "sum"),
    sell_vol=("sell_vol", "sum"),
    abs_x_mean=("abs_x", "mean"),
    sweep_frac=("sweep", "mean"),
)

# 5. 比例特征（分母为 0 时是 NaN，LightGBM 可以处理）
side_feats["buy_frac"] = side_feats["n_buy"] / (side_feats["n_buy"] + side_feats["n_sell"])
side_feats["buy_vol_frac"] = side_feats["buy_vol"] / (side_feats["buy_vol"] + side_feats["sell_vol"])

side_feats = side_feats[["buy_frac", "buy_vol_frac", "abs_x_mean", "sweep_frac"]]

del tq, x, mid, hs, mid_prev, hs_prev, pos, sid, tid, secs, same_window
side_feats


找不到对应 book 行的成交数: 1


buy_frac  buy_vol_frac  abs_x_mean  sweep_frac
stock_id time_id                                                
0        5        0.647059      0.567679    0.902439    0.000000
         11       0.590909      0.458834    0.760597    0.000000
         16       0.200000      0.223513    0.902782    0.040000
         31       0.363636      0.416537    0.912570    0.066667
         62       0.588235      0.392576    0.888332    0.045455
...                    ...           ...         ...         ...
126      32751    0.413793      0.435189    0.813497    0.000000
         32753    0.722222      0.467582    0.909978    0.046512
         32758    0.354839      0.109877    0.965501    0.028571
         32763    0.383562      0.617441    1.044486    0.075000
         32767    0.387097      0.331271    0.961673    0.027778

[428913 rows x 4 columns]

In [25]:
features = []

for feat in ["spread", "wap1", "wap2", "wap12", "tot_size", "log_ret1", "log_ret12"]:
    features.append(window_stats(book, feat))

for feat in ["size", "price", "log_ret"]:
    features.append(window_stats(trade, feat, prefix = "trade_"))

features.extend([imb_features, fwap1_to1, lwap1_to1, book_rv, trade_ratio, trade_rv, trade_tot_volume, side_feats])

features_df = pd.concat(features, axis = 1).reset_index()

features_df = add_market_features(
    features_df,
    {"rv1": "log", "rv2": "log", "trade_rv": "log1p", "n_quotes": "log", "n_trades": "log1p"},
    stats=("mean", "std"),
)

features_df.to_pickle(data_dir + "features1.pkl")

In [26]:
features_df

,stock_id,time_id,spread_mean,spread_std,spread_max,spread_zscore_last,wap1_mean,wap1_std,wap1_max,wap1_zscore_last,...,rel_log_rv2,mkt_mean_log1p_trade_rv,mkt_std_log1p_trade_rv,rel_log1p_trade_rv,mkt_mean_log_n_quotes,mkt_std_log_n_quotes,rel_log_n_quotes,mkt_mean_log1p_n_trades,mkt_std_log1p_n_trades,rel_log1p_n_trades
0,0,5,0.000855,0.000212,0.001396,-1.106809,1.003725,0.000692,1.004920,0.008442,...,0.272819,0.002671,0.000940,-6.674761e-04,6.063915,0.285152,-0.353488,4.446310,0.644525,-0.732738
1,0,11,0.000394,0.000157,0.000903,0.686592,1.000239,0.000261,1.000834,0.255876,...,-0.105863,0.001330,0.000745,-4.293454e-04,5.684519,0.394147,-0.386202,3.769704,0.629037,-0.335717
2,0,16,0.000725,0.000163,0.001149,-1.801277,0.999542,0.000862,1.000878,-2.248671,...,0.437048,0.001540,0.000550,4.191123e-04,5.801210,0.393754,-0.564768,3.967295,0.692018,-0.709199
3,0,31,0.000859,0.000278,0.001619,1.068402,0.998832,0.000754,1.000412,-1.778458,...,0.120077,0.001559,0.000741,4.509930e-07,5.619833,0.473190,-0.832342,3.664575,0.672762,-0.891987
4,0,62,0.000397,0.000130,0.000792,-0.185681,0.999619,0.000257,1.000159,1.784543,...,0.193852,0.001221,0.000538,-3.502905e-04,5.647106,0.453304,-0.476622,3.630579,0.728889,-0.495085
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
428927,126,32751,0.000877,0.000234,0.001567,0.436031,0.999582,0.000485,1.000736,0.050357,...,0.471782,0.001803,0.000729,3.661989e-04,5.943707,0.329720,-0.207134,4.175530,0.608478,-0.537943
428928,126,32753,0.000707,0.000228,0.001379,2.489258,1.002476,0.001262,1.006166,2.563756,...,0.635456,0.001418,0.000866,7.602312e-04,5.805000,0.361466,-0.397828,3.973947,0.681130,-0.189757
428929,126,32758,0.000740,0.000239,0.001284,1.859070,1.001082,0.000465,1.002656,3.386859,...,0.633404,0.001490,0.000464,4.288341e-04,5.655585,0.477700,-0.110408,3.675618,0.695890,-0.092099
428930,126,32763,0.000531,0.000172,0.001119,1.124899,1.001809,0.000456,1.002963,-2.174301,...,0.299680,0.002103,0.000782,-5.420670e-05,6.059868,0.245487,-0.070907,4.595099,0.534508,-0.200650
